# 01: Data Extraction

Extract admission data from **MIMIC-IV v3.1** and prepare for analysis.

**Data Source:** MIMIC-IV v3.1 (https://physionet.org/content/mimiciv/3.1/)  
**Period:** 2008-2019 (dates are shifted ~100 years for privacy - actual dates will appear as 2105-2214)  
**Expected:** 400,000+ admissions

**Note:** MIMIC-IV shifts all dates by approximately 100 years for HIPAA compliance. The relative dates and patterns are preserved, which is sufficient for time series forecasting.

**Setup:**
1. Download `admissions.csv.gz` from: https://physionet.org/content/mimiciv/3.1/
2. Download `patients.csv.gz` from: https://physionet.org/content/mimiciv/3.1/ (required for anchor year alignment)
3. Upload both files using the cell below (or upload to Google Drive manually)


In [ ]:
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Setup paths - both Google Drive (for persistence) and local Colab (for next notebook)
DRIVE_ROOT = '/content/drive/MyDrive/hospital_admissions_forecasting'
DRIVE_DATA_RAW = os.path.join(DRIVE_ROOT, 'data', 'raw')

# Local Colab paths (matching 02_exploratory_analysis.ipynb)
PROJECT_ROOT = '/content/hospital_admissions_forecasting'
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
DATA_RAW = os.path.join(DATA_DIR, 'raw')

# Create directories
for path in [DRIVE_DATA_RAW, DATA_RAW]:
    os.makedirs(path, exist_ok=True)

print("✓ Setup complete")
print(f"  Google Drive: {DRIVE_DATA_RAW}")
print(f"  Local Colab: {DATA_RAW}")


## Upload CSV File (if needed)

The cell below will check if a CSV file already exists. If found, you can skip the upload. Otherwise, upload your `admissions.csv` or `admissions.csv.gz` file.


In [ ]:
# Check if CSV file already exists before prompting for upload
csv_file_drive = os.path.join(DRIVE_DATA_RAW, 'admissions.csv')
csv_gz_file_drive = os.path.join(DRIVE_DATA_RAW, 'admissions.csv.gz')
csv_file_local = os.path.join(DATA_RAW, 'admissions.csv')
csv_gz_file_local = os.path.join(DATA_RAW, 'admissions.csv.gz')
clean_file_drive = os.path.join(DRIVE_DATA_RAW, 'admissions_clean.csv')
clean_file_local = os.path.join(DATA_RAW, 'admissions_clean.csv')

# Check all possible file locations
existing_files = []
if os.path.exists(csv_gz_file_drive):
    size = os.path.getsize(csv_gz_file_drive) / (1024**2)
    existing_files.append(f"✓ Found: {csv_gz_file_drive} ({size:.2f} MB)")
if os.path.exists(csv_gz_file_local):
    size = os.path.getsize(csv_gz_file_local) / (1024**2)
    existing_files.append(f"✓ Found: {csv_gz_file_local} ({size:.2f} MB)")
if os.path.exists(csv_file_drive):
    size = os.path.getsize(csv_file_drive) / (1024**2)
    existing_files.append(f"✓ Found: {csv_file_drive} ({size:.2f} MB)")
if os.path.exists(csv_file_local):
    size = os.path.getsize(csv_file_local) / (1024**2)
    existing_files.append(f"✓ Found: {csv_file_local} ({size:.2f} MB)")
if os.path.exists(clean_file_drive):
    size = os.path.getsize(clean_file_drive) / (1024**2)
    existing_files.append(f"✓ Found: {clean_file_drive} ({size:.2f} MB)")
if os.path.exists(clean_file_local):
    size = os.path.getsize(clean_file_local) / (1024**2)
    existing_files.append(f"✓ Found: {clean_file_local} ({size:.2f} MB)")

if existing_files:
    print("✅ CSV file(s) already found! No need to upload.")
    print("\nExisting files:")
    for file_info in existing_files:
        print(f"  {file_info}")
    print("\n✅ You can skip this cell and proceed to the 'Load Data' cell below.")
else:
    # No file found, prompt for upload
    from google.colab import files
    print("📤 No CSV file found. Please upload your admissions.csv or admissions.csv.gz file...")
    uploaded = files.upload()

    if uploaded:
        uploaded_filename = list(uploaded.keys())[0]
        print(f"\n✓ Uploaded: {uploaded_filename}")
        print(f"  Size: {len(uploaded[uploaded_filename]) / (1024**2):.2f} MB")

        # Save to Google Drive
        if uploaded_filename.endswith('.gz'):
            target_path = os.path.join(DRIVE_DATA_RAW, 'admissions.csv.gz')
        else:
            target_path = os.path.join(DRIVE_DATA_RAW, 'admissions.csv')

        with open(target_path, 'wb') as f:
            f.write(uploaded[uploaded_filename])

        print(f"✓ Saved to Google Drive: {target_path}")
    else:
        print("⚠️  No file uploaded. Please upload the file or place it manually in:")
        print(f"  {DRIVE_DATA_RAW}/admissions.csv.gz")
        print(f"  OR")
        print(f"  {DATA_RAW}/admissions.csv.gz")


In [ ]:
# Load data from CSV file (try multiple locations)
file_paths = [
    (DRIVE_DATA_RAW, 'admissions.csv.gz', True),
    (DATA_RAW, 'admissions.csv.gz', True),
    (DRIVE_DATA_RAW, 'admissions.csv', False),
    (DATA_RAW, 'admissions.csv', False),
]

admissions = None
for folder, filename, is_gzip in file_paths:
    file_path = os.path.join(folder, filename)
    if os.path.exists(file_path):
        try:
            if is_gzip:
                admissions = pd.read_csv(file_path, parse_dates=['admittime', 'dischtime'], 
                                        low_memory=False, compression='gzip')
            else:
                admissions = pd.read_csv(file_path, parse_dates=['admittime', 'dischtime'], 
                                        low_memory=False)
            print(f"✓ Loaded {len(admissions):,} records from {file_path}")
            break
        except Exception as e:
            print(f"⚠️  Error loading {file_path}: {e}")
            continue

if admissions is None or len(admissions) == 0:
    print("❌ CSV file not found! Upload admissions.csv.gz to Google Drive or local Colab.")
else:
    print(f"Date range: {admissions['admittime'].min()} to {admissions['admittime'].max()}")
    print(f"  (Note: MIMIC-IV shifts dates ~100 years for privacy)")


## Data Exploration and Quality Checks


In [ ]:
# Quick data overview
if admissions is not None and len(admissions) > 0:
    print(f"Shape: {admissions.shape}")
    print(f"Missing values: {admissions.isnull().sum().sum()} total")
else:
    print("⚠️  No data loaded. Please run the 'Load Data' cell above first.")


## Data Cleaning


In [ ]:
# Clean data
if admissions is not None and len(admissions) > 0:
    initial_count = len(admissions)
    print(f"Starting with {initial_count:,} records")

    # Convert dates
    admissions['admittime'] = pd.to_datetime(admissions['admittime'], errors='coerce')
    admissions['dischtime'] = pd.to_datetime(admissions['dischtime'], errors='coerce')

    # Remove invalid records
    admissions = admissions.dropna(subset=['admittime'])  # Missing admission time
    admissions = admissions.drop_duplicates(subset=['hadm_id'], keep='first')  # Duplicate IDs
    admissions = admissions[admissions['dischtime'].isna() | (admissions['dischtime'] >= admissions['admittime'])]  # Invalid discharge dates

    # Calculate length of stay
    if 'dischtime' in admissions.columns:
        admissions['length_of_stay'] = (admissions['dischtime'] - admissions['admittime']).dt.days
        admissions = admissions[admissions['length_of_stay'].isna() | (admissions['length_of_stay'] >= 0)]

    # Sort by admission time
    admissions_clean = admissions.sort_values('admittime').reset_index(drop=True)

    print(f"✓ Cleaning complete: {len(admissions_clean):,} records remaining")
    print(f"  Removed: {initial_count - len(admissions_clean):,} records")
    print(f"Date range: {admissions_clean['admittime'].min()} to {admissions_clean['admittime'].max()}")
else:
    print("⚠️  No data loaded")
    admissions_clean = pd.DataFrame()


## Data Quality Summary


In [ ]:
# Data quality summary
if len(admissions_clean) > 0:
    print(f"📊 Total records: {len(admissions_clean):,}")
    print(f"   Unique subjects: {admissions_clean['subject_id'].nunique():,}")
    if 'length_of_stay' in admissions_clean.columns:
        los = admissions_clean['length_of_stay'].dropna()
        if len(los) > 0:
            print(f"   Length of Stay: Mean {los.mean():.1f} days, Median {los.median():.1f} days")
    print("✅ Data ready for anchor year alignment")
else:
    print("⚠️  No cleaned data available")


## Save Cleaned Data

**Important:** This cell should be run AFTER the "Anchor Year Alignment" cell below (Cell 14) to ensure the `aligned_date` and `anchor_year_group` columns are included in the saved data.


## Anchor Year Alignment

MIMIC-IV dates are deidentified by shifting them per-patient (not uniformly). To properly aggregate admissions by date, we need to align each admission to its patient's anchor year.

**Note:** The `anchor_year_group` provides a 3-year range (e.g., "2011 - 2013"), so exact dates are approximate within that range.

**Default Approach:** Dates are aligned to the anchor year group and left in the 3-year range for averaging in the next notebook.

**Optional Enhancement:** Set `ENABLE_SPECIFIC_DATE_IDENTIFICATION = True` in Cell 14 to enable:
- **Leap year detection**: Feb 29 dates can uniquely identify leap years
- **Day-of-week identification**: Uses preserved day-of-week to identify unique years when possible
- For ambiguous cases, dates remain in the 3-year range for averaging

This creates the `aligned_date` column which is used for aggregation in `02_exploratory_analysis.ipynb`.


In [ ]:
# Save cleaned data AFTER anchor year alignment
# This cell should be run AFTER the anchor year alignment cell below
if len(admissions_clean) > 0:
    output_file_drive = os.path.join(DRIVE_DATA_RAW, 'admissions_clean.csv')
    output_file_local = os.path.join(DATA_RAW, 'admissions_clean.csv')

    admissions_clean.to_csv(output_file_drive, index=False)
    admissions_clean.to_csv(output_file_local, index=False)

    file_size = os.path.getsize(output_file_local) / (1024**2)
    print(f"✓ Saved to Google Drive: {output_file_drive}")
    print(f"✓ Saved to local Colab: {output_file_local}")
    print(f"  Size: {file_size:.2f} MB, Records: {len(admissions_clean):,}")
    
    if 'aligned_date' in admissions_clean.columns and 'anchor_year_group' in admissions_clean.columns:
        print(f"  ✓ Includes 'aligned_date' and 'anchor_year_group' for HYBRID day-of-week aggregation")
    print(f"\n✅ Data ready for next notebook (02_exploratory_analysis.ipynb)")
else:
    print("⚠️  No cleaned data available")


In [ ]:
# Anchor Year Alignment
# ============================================================================
# CONFIGURATION: Specific Date Identification
# ============================================================================
# Set to True to enable day-of-week + leap year identification for unique dates
# Set to False to use simple averaging approach (dates remain in 3-year range)
ENABLE_SPECIFIC_DATE_IDENTIFICATION = False  # Default: False (use averaging)

if len(admissions_clean) > 0:
    print("📅 Aligning dates using anchor year information...")
    if ENABLE_SPECIFIC_DATE_IDENTIFICATION:
        print("  ⚙️  Specific date identification: ENABLED (day-of-week + leap year)")
    else:
        print("  ⚙️  Specific date identification: DISABLED (using averaging approach)")

    # Load patients table
    patients_paths = [
        os.path.join(DRIVE_DATA_RAW, 'patients.csv.gz'),
        os.path.join(DATA_RAW, 'patients.csv.gz'),
        os.path.join(DRIVE_DATA_RAW, 'patients.csv'),
        os.path.join(DATA_RAW, 'patients.csv')
    ]
    
    patients_df = None
    for file_path in patients_paths:
        if os.path.exists(file_path):
            try:
                patients_df = pd.read_csv(file_path, compression='gzip' if file_path.endswith('.gz') else None, 
                                        low_memory=False)
                print(f"✓ Loaded patients table: {len(patients_df):,} records")
                break
            except Exception as e:
                continue

    if patients_df is not None:
        # Check required columns
        required_cols = ['subject_id', 'anchor_year', 'anchor_year_group']
        missing_cols = [col for col in required_cols if col not in patients_df.columns]

        if missing_cols:
            print(f"⚠️  Missing required columns in patients table: {missing_cols}")
            print("   Proceeding without anchor year alignment (using deidentified dates as-is)")
            admissions_clean['aligned_date'] = admissions_clean['admittime']
        else:
            # Select only needed columns to reduce memory
            patients_subset = patients_df[required_cols].copy()

            # Join admissions with patients on subject_id
            print(f"  Joining admissions with patients table...")
            admissions_with_anchor = admissions_clean.merge(
                patients_subset,
                on='subject_id',
                how='left'
            )

            # Check how many admissions have anchor year info
            has_anchor = admissions_with_anchor['anchor_year'].notna().sum()
            print(f"  Admissions with anchor year: {has_anchor:,} ({has_anchor/len(admissions_with_anchor)*100:.1f}%)")

            # Calculate year offset for each admission
            # The deidentified year minus the anchor year gives us the offset
            admissions_with_anchor['admittime_year'] = pd.to_datetime(admissions_with_anchor['admittime']).dt.year
            admissions_with_anchor['year_offset'] = (
                admissions_with_anchor['admittime_year'] - admissions_with_anchor['anchor_year']
            )

            # Create aligned date: keep month/day from deidentified date, use anchor_year + offset
            # This gives us an approximate real date within the anchor_year_group
            def create_aligned_date(row):
                if pd.isna(row['anchor_year']) or pd.isna(row['year_offset']):
                    # If no anchor year, use original date
                    return row['admittime']

                try:
                    admittime = pd.to_datetime(row['admittime'])
                    # Parse anchor_year_group to get the real year range
                    # Format: "2011 - 2013" -> start_year = 2011, end_year = 2013
                    anchor_group_start = None
                    anchor_group_end = None
                    if 'anchor_year_group' in row and not pd.isna(row.get('anchor_year_group')):
                        try:
                            parts = str(row['anchor_year_group']).split(' - ')
                            if len(parts) == 2:
                                anchor_group_start = int(parts[0].strip())
                                anchor_group_end = int(parts[1].strip())
                        except:
                            pass

                    if anchor_group_start is None or anchor_group_end is None:
                        # If we can't parse anchor_year_group, fall back to original date
                        return row['admittime']

                    # Calculate real year: anchor_group_start + offset
                    # Example: anchor_year=2158, anchor_group="2011 - 2013", admittime_year=2158
                    # offset = 2158 - 2158 = 0, real_year = 2011 + 0 = 2011
                    real_year = anchor_group_start + int(row['year_offset'])

                    # CRITICAL FIX: Clamp real_year to anchor_year_group range
                    # This prevents dates from going outside the 3-year range when admissions
                    # occur far from the anchor_year (creating extreme offsets)
                    # Example: If offset pushes real_year to 2096, clamp it to 2010 (end of "2008 - 2010")
                    real_year = max(anchor_group_start, min(anchor_group_end, real_year))

                    # Create aligned date with same month/day, aligned year
                    aligned_date = pd.Timestamp(
                        year=real_year,
                        month=admittime.month,
                        day=admittime.day,
                        hour=admittime.hour,
                        minute=admittime.minute,
                        second=admittime.second
                    )
                    return aligned_date
                except:
                    # Fallback to original date if alignment fails
                    return row['admittime']

            print(f"  Calculating aligned dates...")
            admissions_with_anchor['aligned_date'] = admissions_with_anchor.apply(create_aligned_date, axis=1)

            # Calculate real_year for diagnostic purposes
            admissions_with_anchor['real_year'] = admissions_with_anchor['aligned_date'].dt.year

            # Report on date clamping
            if 'anchor_year_group' in admissions_with_anchor.columns:
                clamped_count = 0
                for _, row in admissions_with_anchor.iterrows():
                    if pd.notna(row.get('anchor_year_group')) and pd.notna(row.get('year_offset')):
                        try:
                            parts = str(row['anchor_year_group']).split(' - ')
                            if len(parts) == 2:
                                start = int(parts[0].strip())
                                end = int(parts[1].strip())
                                unclamped_year = start + int(row['year_offset'])
                                if unclamped_year < start or unclamped_year > end:
                                    clamped_count += 1
                        except:
                            pass
                if clamped_count > 0:
                    print(f"  ⚠️  {clamped_count:,} dates ({clamped_count/len(admissions_with_anchor)*100:.1f}%) clamped to anchor_year_group bounds")

            # Update admissions_clean with aligned_date
            admissions_clean = admissions_with_anchor.copy()

            # Report date range comparison
            original_min = admissions_clean['admittime'].min()
            original_max = admissions_clean['admittime'].max()
            aligned_min = admissions_clean['aligned_date'].min()
            aligned_max = admissions_clean['aligned_date'].max()

            print(f"\n📊 Date Range Comparison:")
            print(f"  Original (deidentified): {original_min.date()} to {original_max.date()}")
            print(f"  Aligned (approximate):   {aligned_min.date()} to {aligned_max.date()}")

            # Validate date range
            aligned_years = admissions_clean['aligned_date'].dt.year
            outside_range = ((aligned_years < 2008) | (aligned_years > 2022)).sum()
            if outside_range > 0:
                print(f"\n  ⚠️  {outside_range:,} admissions ({outside_range/len(admissions_clean)*100:.2f}%) outside expected range (2008-2022)")

            # Show anchor year group distribution
            if 'anchor_year_group' in admissions_clean.columns:
                year_groups = admissions_clean['anchor_year_group'].value_counts().sort_index()
                print(f"\n📅 Anchor Year Group Distribution:")
                for group, count in year_groups.items():
                    pct = count / len(admissions_clean) * 100
                    print(f"  {group}: {count:,} ({pct:.1f}%)")

            print(f"\n✓ Date alignment complete.")
            
            # OPTIONAL: Specific date identification (day-of-week + leap year)
            if ENABLE_SPECIFIC_DATE_IDENTIFICATION:
                print("\n🔍 Applying specific date identification (day-of-week + leap year)...")
                
                def identify_year_from_day_of_week(month, day, day_of_week, start_year, end_year):
                    """Identify which year(s) in the range match the day-of-week.
                    
                    Enhanced with leap year detection: Feb 29 dates can uniquely identify leap years.
                    """
                    # LEAP YEAR DETECTION: Feb 29 must be a leap year
                    if month == 2 and day == 29:
                        # Find all leap years in the range
                        leap_years = [y for y in range(start_year, end_year + 1) 
                                     if y % 4 == 0 and (y % 100 != 0 or y % 400 == 0)]
                        
                        if len(leap_years) == 1:
                            # Unique leap year in range - verify day-of-week matches
                            try:
                                date = pd.Timestamp(year=leap_years[0], month=2, day=29)
                                if date.dayofweek == day_of_week:
                                    return leap_years[0]  # Perfect match!
                            except:
                                pass
                        elif len(leap_years) > 1:
                            # Multiple leap years - check day-of-week for each
                            matching_leap_years = []
                            for year in leap_years:
                                try:
                                    date = pd.Timestamp(year=year, month=2, day=29)
                                    if date.dayofweek == day_of_week:
                                        matching_leap_years.append(year)
                                except:
                                    pass
                            if len(matching_leap_years) == 1:
                                return matching_leap_years[0]
                            elif len(matching_leap_years) > 1:
                                return matching_leap_years  # Ambiguous
                        # If no leap years or no match, fall through to regular logic
                    
                    # REGULAR DAY-OF-WEEK MATCHING
                    matching_years = []
                    for year in range(start_year, end_year + 1):
                        try:
                            date = pd.Timestamp(year=year, month=month, day=day)
                            if date.dayofweek == day_of_week:
                                matching_years.append(year)
                        except ValueError:
                            # Handle invalid dates (e.g., Feb 29 in non-leap years)
                            if month == 2 and day == 29:
                                # Skip non-leap years for Feb 29
                                continue
                            # For other invalid dates, try Feb 28 as fallback
                            try:
                                date = pd.Timestamp(year=year, month=2, day=28)
                                if date.dayofweek == day_of_week:
                                    matching_years.append(year)
                            except:
                                pass
                            continue
                    
                    if len(matching_years) == 1:
                        return matching_years[0]  # Unique match!
                    elif len(matching_years) > 1:
                        return matching_years  # Ambiguous
                    else:
                        return None
                
                # Extract day-of-week from deidentified date (preserved)
                admittime = pd.to_datetime(admissions_clean['admittime'])
                admissions_clean['month'] = admittime.dt.month
                admissions_clean['day'] = admittime.dt.day
                admissions_clean['day_of_week'] = admittime.dt.dayofweek  # 0=Monday, 6=Sunday
                
                # Refine aligned_date using day-of-week (vectorized where possible)
                unique_count = 0
                ambiguous_count = 0
                
                # Filter to rows with valid anchor_year_group and aligned_date
                valid_mask = admissions_clean['anchor_year_group'].notna() & admissions_clean['aligned_date'].notna()
                valid_rows = admissions_clean[valid_mask].copy()
                
                if len(valid_rows) > 0:
                    # Parse anchor_year_group once
                    def parse_group(group_str):
                        try:
                            parts = str(group_str).split(' - ')
                            if len(parts) == 2:
                                return int(parts[0].strip()), int(parts[1].strip())
                        except:
                            pass
                        return None, None
                    
                    # Process in batches for better performance
                    for idx, row in valid_rows.iterrows():
                        start_year, end_year = parse_group(row['anchor_year_group'])
                        if start_year is None:
                            continue
                        
                        # Identify year using day-of-week
                        year_match = identify_year_from_day_of_week(
                            int(row['month']), int(row['day']),
                            int(row['day_of_week']), start_year, end_year
                        )
                        
                        if isinstance(year_match, int):
                            # Unique match! Use this specific year
                            unique_count += 1
                            try:
                                original_date = pd.to_datetime(row['aligned_date'])
                                admissions_clean.at[idx, 'aligned_date'] = pd.Timestamp(
                                    year=year_match,
                                    month=int(row['month']),
                                    day=int(row['day']),
                                    hour=original_date.hour,
                                    minute=original_date.minute,
                                    second=original_date.second
                                )
                            except:
                                pass
                        elif isinstance(year_match, list):
                            # Ambiguous - keep original aligned_date
                            ambiguous_count += 1
                
                # Clean up temporary columns before saving
                temp_cols_to_remove = ['month', 'day', 'day_of_week', 'admittime_year', 'year_offset', 'real_year']
                for col in temp_cols_to_remove:
                    if col in admissions_clean.columns:
                        admissions_clean = admissions_clean.drop(columns=[col])
                
                total = len(admissions_clean)
                pct_unique = (unique_count / total * 100) if total > 0 else 0
                pct_ambiguous = (ambiguous_count / total * 100) if total > 0 else 0
                print(f"  ✓ Day-of-week identification: {unique_count:,} unique ({pct_unique:.1f}%), {ambiguous_count:,} ambiguous ({pct_ambiguous:.1f}%)")
                print(f"  ✓ Using 'aligned_date' column (refined with day-of-week) for aggregation")
            else:
                # Default: Leave dates in 3-year range for averaging
                print("\n  ℹ️  Dates left in 3-year range for averaging (specific date identification disabled)")
                print(f"  ✓ Using 'aligned_date' column (within anchor_year_group range) for aggregation")
                # Clean up temporary columns (but keep aligned_date and anchor_year_group)
                temp_cols_to_remove = ['admittime_year', 'year_offset', 'real_year']
                for col in temp_cols_to_remove:
                    if col in admissions_clean.columns:
                        admissions_clean = admissions_clean.drop(columns=[col])
    else:
        print("⚠️  Patients table not found. Proceeding without anchor year alignment.")
        print(f"   Using deidentified dates as-is (relative patterns preserved)")
        admissions_clean['aligned_date'] = admissions_clean['admittime']
else:
    print("⚠️  No cleaned admissions data available for date alignment")


## Summary

This notebook extracts and prepares admission data from **MIMIC-IV v3.1**. Key outputs:
- Cleaned admission records with timestamps
- **`aligned_date` column**: Dates aligned to anchor year groups (within 3-year range for averaging)
- **`anchor_year_group` column**: 3-year ranges for each patient (e.g., "2011 - 2013")
- Data quality documentation
- Ready for aggregation in next notebook

**Configuration:** By default, dates are left in the 3-year range for averaging. Set `ENABLE_SPECIFIC_DATE_IDENTIFICATION = True` in Cell 14 to enable day-of-week + leap year identification.

**Data Source:** MIMIC-IV v3.1 (https://physionet.org/content/mimiciv/3.1/)

**Next Steps:** Proceed to `02_exploratory_analysis.ipynb` to aggregate and explore the data. The `aligned_date` column will be used for daily aggregation.
